# 02 — Feature Engineering Temporal (Fase 2B)

Para cada variable y paciente: valor actual, media, máx, mín, varianza, std, **pendiente**, **aceleración**, % de cambio, cambio acumulado y tiempo desde la última medición. Y los mismos estadísticos en **ventanas rolling de 7 / 30 / 90 / 180 días**.

In [1]:
# --- Setup ---
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

TCF = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(TCF / "src"))
from predia_temporal import config
DS, MET, FIG, DASH = TCF/"datasets", TCF/"metrics", TCF/"figures", TCF/"dashboards"
def show(p, w=None): display(Image(filename=str(p), width=w))
def load(j): return json.load(open(j))
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 160)
print("Entorno listo:", TCF)


Entorno listo: /home/wowo/Descargas/predia/ml-research/temporal_clinical_framework


In [2]:
feat = pd.read_csv(DS/'features.csv')
print('Matriz de features:', feat.shape)
gl = [c for c in feat.columns if c.startswith('glucosa_')]
print('\nFeatures de glucosa (%d):' % len(gl)); print(gl)

Matriz de features: (400, 183)

Features de glucosa (30):
['glucosa_current', 'glucosa_mean', 'glucosa_max', 'glucosa_min', 'glucosa_var', 'glucosa_std', 'glucosa_cv', 'glucosa_slope_m', 'glucosa_r2', 'glucosa_accel_m2', 'glucosa_pct_change', 'glucosa_cum_change', 'glucosa_time_since_last', 'glucosa_n_obs', 'glucosa_w7_mean', 'glucosa_w7_slope_m', 'glucosa_w7_std', 'glucosa_w7_n', 'glucosa_w30_mean', 'glucosa_w30_slope_m', 'glucosa_w30_std', 'glucosa_w30_n', 'glucosa_w90_mean', 'glucosa_w90_slope_m', 'glucosa_w90_std', 'glucosa_w90_n', 'glucosa_w180_mean', 'glucosa_w180_slope_m', 'glucosa_w180_std', 'glucosa_w180_n']


## Ventanas rolling: la pendiente reciente vs la histórica
Comparar `slope_m` (toda la serie) con `w30_slope_m` (últimos 30 d) revela **aceleraciones o reversiones** recientes de la tendencia.

In [3]:
cols = ['glucosa_slope_m','glucosa_w30_slope_m','imc_slope_m','imc_w30_slope_m','riesgo_current']
feat.groupby('archetype')[cols].mean().round(3)  # features.csv ya incluye 'archetype'

,glucosa_slope_m,glucosa_w30_slope_m,imc_slope_m,imc_w30_slope_m,riesgo_current
archetype,,,,,
Alto riesgo persistente,2.986,-1.500,0.051,0.039,0.977
Deterioro lento,7.336,5.000,0.201,0.148,0.896
Estable,0.897,-1.120,0.001,0.144,0.239
Mejora rápida,-5.833,-6.475,-0.347,-0.430,0.062
Oscilante,1.374,-4.238,-0.005,0.267,0.683


### Conclusión 2B
Cada paciente queda descrito por un vector de ~180 features temporales que capturan nivel, dispersión, dirección y dinámica reciente.